# Module 02 — Fine-tuning with LoRA

In this notebook you will learn:
- Why fine-tuning matters (and when NOT to use it)
- What LoRA is and why it's efficient
- How to prepare a dataset
- How to fine-tune a model and save it


## 1. Why Fine-tune?

Pre-trained models know a lot, but they're generalists. Fine-tuning teaches the model:
- Your domain-specific vocabulary and style
- How to respond in a specific format
- Knowledge that wasn't in the original training data

**When to use RAG instead of fine-tuning:**
- Data changes frequently → use RAG (Module 03)
- You need the model to *behave* differently → use fine-tuning
- You need the model to *know* new facts → RAG is usually better


## 2. What is LoRA?

Full fine-tuning updates all model weights — expensive and slow.

**LoRA (Low-Rank Adaptation)** freezes the original weights and adds small trainable matrices alongside them. This means:
- 100x fewer trainable parameters
- Can run on a laptop CPU (with a small model)
- Easy to swap between different fine-tuned "adapters"


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import torch

model_name = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,              # rank — higher = more capacity but more params
    lora_alpha=32,    # scaling factor
    lora_dropout=0.1,
    target_modules=['c_attn']  # which layers to adapt (GPT-2 style)
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Prepare Training Data

For this example we'll create a tiny dataset. In practice, you'd load real data from files or HuggingFace datasets.


In [ ]:
# Example: teaching the model to answer in a specific style
training_examples = [
    {'text': 'Question: What is RAG? Answer: RAG stands for Retrieval-Augmented Generation. It connects an LLM to an external knowledge base so it can answer questions with up-to-date information.'},
    {'text': 'Question: What is LoRA? Answer: LoRA is Low-Rank Adaptation, a technique that fine-tunes only a small number of additional parameters instead of all model weights.'},
    {'text': 'Question: How do you keep an LLM up to date? Answer: You can use RAG with a regularly updated vector database, or periodically fine-tune the model on new data.'},
    {'text': 'Question: What is a context window? Answer: A context window is the maximum number of tokens an LLM can process at once. Text outside this window is not visible to the model.'},
]

def tokenize(example):
    return tokenizer(
        example['text'],
        truncation=True,
        max_length=128,
        padding='max_length'
    )

dataset = Dataset.from_list(training_examples)
tokenized = dataset.map(tokenize, remove_columns=['text'])
tokenized = tokenized.map(lambda x: {'labels': x['input_ids']})

print(f'Dataset size: {len(tokenized)} examples')

## 4. Train


In [ ]:
training_args = TrainingArguments(
    output_dir='./models/lora-distilgpt2',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy='epoch',
    report_to='none'  # disable wandb for now
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
)

trainer.train()

## 5. Save and Load the Adapter


In [ ]:
# Save only the LoRA adapter (small file, not the full model)
model.save_pretrained('./models/lora-distilgpt2-adapter')
tokenizer.save_pretrained('./models/lora-distilgpt2-adapter')
print('Adapter saved!')

In [ ]:
# Load the adapter back on top of the base model
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained('distilgpt2')
loaded_model = PeftModel.from_pretrained(base_model, './models/lora-distilgpt2-adapter')

from transformers import pipeline
gen = pipeline('text-generation', model=loaded_model, tokenizer=tokenizer, device='cpu')
print(gen('Question: What is RAG? Answer:', max_new_tokens=60, do_sample=False)[0]['generated_text'])

## Next Steps
Move on to `03_rag/` to learn how to connect your model to a live knowledge base.
